# Day 06 Lab 0: 데이터셋 구축 — 공개 데이터 확장 + 합성 데이터 생성

## 목적
Lab 1(SFT)과 Lab 2(Embedding Tuning)에서 사용할 도메인 특화 데이터셋을 구축

## 2-Track 접근법

```
Track A: 공개 한국어 데이터셋 다운로드 → 도메인 필터링     (API 불필요)
Track B: OpenAI API로 도메인 합성 데이터 생성              (API 필요)
                    ↓                        ↓
              ┌─────────────────────────────────────┐
              │    혼합 + 품질 검증 + 저장            │
              └─────────────────────────────────────┘
```

| Track | 필요 조건 | 산출물 |
|-------|----------|--------|
| **Track A만** | `pip install datasets` | 공개 데이터 필터링 결과 (도메인 인접 샘플) |
| **Track B만** | `OPENAI_API_KEY` | 도메인 전용 합성 데이터 |
| **A + B 혼합** (권장) | 둘 다 | 일반 한국어 능력 + 도메인 전문성 |

> **핵심 원칙:** 공개 데이터셋은 "폴백"이 아니라 **학습의 기반(base)**이다.
> 일반 한국어 응답 능력이 없으면 도메인 답변도 부자연스러워진다.

In [ ]:
import os
import json
import time
import random
from pathlib import Path

import pandas as pd
from openai import OpenAI

SEED = 3407
random.seed(SEED)
print("Imports OK")

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────
SEED_DOCS = {
    "observability_overview": Path("../Day-05/AI Agent Observability Overview.md"),
    "operations_runbook": Path("../Day-05/docs/operations_runbook.md"),
    "sla_doc": Path("../Day-05/project/data/corpus/00_sla.md"),
}

SFT_TRAIN_PATH = Path("sft_train.jsonl")
SFT_EVAL_PATH = Path("sft_eval.jsonl")
EMB_DATA_DIR = Path("data")
EMB_TRAIN_PATH = EMB_DATA_DIR / "embedding_train.jsonl"
EMB_EVAL_PATH = EMB_DATA_DIR / "embedding_eval.jsonl"

# ── 모델 및 생성 파라미터 ────────────────────────────────────────────────────
OPENAI_MODEL = "gpt-5.4-mini"
MAX_SOURCE_CHUNKS = 8
SFT_EXAMPLES_PER_CHUNK = 3
EMB_EXAMPLES_PER_CHUNK = 4
NUM_EVAL_QUERIES = 30

# ── 도메인 키워드 (품질 검사용) ─────────────────────────────────────────────
DOMAIN_KEYWORDS = {
    "metrics": [
        "sla",
        "ttfe",
        "e2e latency",
        "latency",
        "error rate",
        "availability",
        "throughput",
    ],
    "operations": [
        "runbook",
        "incident",
        "triage",
        "remediation",
        "rollback",
        "on-call",
        "alert",
    ],
    "agentops": [
        "agentops",
        "observability",
        "trace",
        "evaluation",
        "prompt optimization",
        "monitoring",
    ],
}

DOMAIN_TASK_TYPES = [
    "metric_explanation",
    "runbook_step_ordering",
    "incident_summary",
    "remediation_suggestion",
    "glossary_definition",
    "alert_triage",
]

EMB_TASK_INSTRUCTIONS = [
    "Retrieve relevant passages about monitoring and observability metrics",
    "Find documentation about incident response and operational procedures",
    "Search for definitions and explanations of SLA and performance terms",
    "Retrieve passages about agent evaluation and prompt optimization",
]

print("Configuration loaded")

---

## Track A: 공개 한국어 데이터셋 다운로드 및 도메인 필터링

> **API 불필요** — HuggingFace 공개 데이터셋만으로 학습 데이터를 구축합니다.

### 왜 공개 데이터셋을 기반으로 쓰는가?

1. **일반 한국어 능력 보존** — 도메인 데이터만으로 학습하면 일반 한국어 응답이 부자연스러워짐
2. **데이터 규모 확보** — 합성 데이터 150~250건은 SFT에 부족. 공개 데이터로 3,000건+ 확보 가능
3. **질문 패턴 다양성** — 인간이 작성한 질문은 합성 질문보다 패턴이 다양함
4. **도메인 인접 샘플** — IT/운영/모니터링 관련 질문이 이미 공개 데이터에 존재함

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Track A-1: SFT용 공개 한국어 데이터셋 다운로드
# ══════════════════════════════════════════════════════════════════════════════
from datasets import load_dataset, concatenate_datasets

print("=" * 60)
print(" SFT 공개 데이터셋 다운로드")
print("=" * 60)

# ── 1. 기본 한국어 instruction 데이터 (Apache 2.0) ──
# 107K rows, 5개 소스 정제 통합, 가장 깨끗한 단일 파일
sft_base = load_dataset("jojo0217/korean_rlhf_dataset", split="train")
print(f"\n[1] jojo0217/korean_rlhf_dataset: {len(sft_base):,} rows")
print(f"    columns: {sft_base.column_names}")
print(f"    license: Apache 2.0")
display(sft_base.to_pandas()[["instruction", "output"]].head(3))

# ── 2. 고품질 인간 작성 QA (네이버 지식인 출처) ──
# 21K rows, 실제 사용자 질문 + 답변
sft_human = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
print(f"\n[2] beomi/KoAlpaca-v1.1a: {len(sft_human):,} rows")
print(f"    columns: {sft_human.column_names}")
print(f"    특징: 네이버 지식인 실제 인간 QA")
display(sft_human.to_pandas()[["instruction", "output"]].head(3))

# ── 3. 다양한 태스크 (Databricks 직원 작성 + DeepL 번역) ──
# 15K rows, 8가지 instruction 카테고리
sft_dolly = load_dataset("nlpai-lab/databricks-dolly-15k-ko", split="train")
print(f"\n[3] nlpai-lab/databricks-dolly-15k-ko: {len(sft_dolly):,} rows")
print(f"    columns: {sft_dolly.column_names}")
print(f"    license: CC-BY-SA 3.0")
print(f"    categories: {set(sft_dolly['category'])}")

print(f"\n총 SFT 후보: {len(sft_base) + len(sft_human) + len(sft_dolly):,} rows")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Track A-2: SFT 도메인 필터링 — IT/운영/모니터링 관련 샘플 추출
# ══════════════════════════════════════════════════════════════════════════════

# 도메인 키워드 (한국어 + 영어)
FILTER_KEYWORDS = [
    # 한국어
    "모니터링",
    "서버",
    "장애",
    "로그",
    "알림",
    "지표",
    "메트릭",
    "배포",
    "인시던트",
    "가용성",
    "트래픽",
    "성능",
    "디버깅",
    "오류",
    "응답 시간",
    "운영",
    "대시보드",
    "알림",
    "에이전트",
    "자동화",
    "파이프라인",
    # 영어 (코드/용어 혼용)
    "api",
    "latency",
    "error rate",
    "sla",
    "monitoring",
    "observability",
    "alert",
    "incident",
    "runbook",
    "trace",
    "log",
    "metric",
    "deploy",
]


def compute_domain_score(example):
    """instruction + output에서 도메인 키워드 매칭 점수를 계산한다."""
    text = f"{example.get('instruction', '')} {example.get('input', '')} {example.get('output', '')}".lower()
    matched = [kw for kw in FILTER_KEYWORDS if kw.lower() in text]
    return {"domain_score": len(matched), "matched_keywords": matched}


# 각 데이터셋에서 도메인 인접 샘플 추출
print("도메인 필터링 중...")

sft_base_scored = sft_base.map(compute_domain_score, desc="Scoring base")
sft_human_scored = sft_human.map(compute_domain_score, desc="Scoring human")
sft_dolly_scored = sft_dolly.map(compute_domain_score, desc="Scoring dolly")

# domain_score >= 2인 샘플 = 키워드 2개 이상 매칭 = 도메인 인접
sft_base_domain = sft_base_scored.filter(lambda x: x["domain_score"] >= 2)
sft_human_domain = sft_human_scored.filter(lambda x: x["domain_score"] >= 2)
sft_dolly_domain = sft_dolly_scored.filter(
    lambda x: x["domain_score"] >= 1
)  # dolly는 기준 완화

print(f"\n도메인 인접 샘플 추출 결과:")
print(
    f"  korean_rlhf:  {len(sft_base_domain):,} / {len(sft_base):,} ({len(sft_base_domain) / len(sft_base):.1%})"
)
print(
    f"  KoAlpaca:     {len(sft_human_domain):,} / {len(sft_human):,} ({len(sft_human_domain) / len(sft_human):.1%})"
)
print(
    f"  dolly-15k-ko: {len(sft_dolly_domain):,} / {len(sft_dolly):,} ({len(sft_dolly_domain) / len(sft_dolly):.1%})"
)

# 도메인 인접 샘플 미리보기
print("\n=== 도메인 인접 샘플 예시 (korean_rlhf) ===")
if len(sft_base_domain) > 0:
    display(
        sft_base_domain.to_pandas()[
            ["instruction", "output", "domain_score", "matched_keywords"]
        ].head(5)
    )

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Track A-3: Embedding용 공개 한국어 데이터셋 다운로드
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print(" Embedding 공개 데이터셋 다운로드")
print("=" * 60)

# ── 1. 한국어 임베딩 전용 트리플릿 (가장 중요!) ──
# 744K rows, 고려대 NLP Lab, KoE5/bge-m3-ko 학습에 사용됨
emb_triplet = load_dataset("nlpai-lab/ko-triplet-v1.0", split="train")
print(f"\n[1] nlpai-lab/ko-triplet-v1.0: {len(emb_triplet):,} rows")
print(f"    columns: {emb_triplet.column_names}")
print(f"    특징: 한국어 임베딩 학습 목적 빌트 (query/document/hard_negative)")
display(emb_triplet.to_pandas()[["query", "document", "hard_negative"]].head(3))

# ── 2. NLI → SimCSE 트리플릿 (사전 변환 완료) ──
# 487K rows, premise/entailment/contradiction 바로 사용 가능
emb_nli = load_dataset("dkoterwa/kor_nli_simcse", split="train")
print(f"\n[2] dkoterwa/kor_nli_simcse: {len(emb_nli):,} rows")
print(f"    columns: {emb_nli.column_names}")
print(f"    license: CC BY-SA 4.0")
print(f"    특징: KakaoBrain NLI에서 SimCSE 트리플릿으로 사전 변환")
display(emb_nli.to_pandas()[["premise", "entailment", "contradiction"]].head(3))

# ── 3. KorQuAD + Hard Negative (검색 학습에 직접 사용) ──
# 147K rows, 질문/문맥/오답 4개 포함
emb_korquad = load_dataset("sungmineom/korquad_negative_samples", split="train")
print(f"\n[3] sungmineom/korquad_negative_samples: {len(emb_korquad):,} rows")
print(f"    columns: {emb_korquad.column_names}")
print(f"    특징: KorQuAD + BM25 기반 hard negative 4개 포함")

print(
    f"\n총 Embedding 후보: {len(emb_triplet) + len(emb_nli) + len(emb_korquad):,} rows"
)

---

### 경로 선택

Track A(공개 데이터)를 완료했습니다. 이제 두 가지 경로 중 선택하세요:

1. **Track B도 실행** (권장) → 아래 "Track B: 합성 데이터 생성" 섹션을 계속 진행
   - OpenAI API로 AgentOps 도메인 전용 데이터를 추가 생성
   - Track A + B를 혼합하여 최종 데이터셋 구축

2. **Track A만으로 진행** → 아래 "Track B" 섹션을 건너뛰고 **"Part 5: 데이터셋 혼합"** 으로 이동
   - 공개 데이터셋의 일반 샘플 + 도메인 필터링 샘플로 구성
   - 도메인 전용 합성 데이터 없이도 Lab 1/2 진행 가능

---

## Track B: 합성 데이터 생성 (OpenAI API)

> **OPENAI_API_KEY 필요** — Track A만으로 진행하려면 이 섹션을 건너뛰세요.

Day-05 seed 문서에서 AgentOps/Observability 도메인 전용 데이터를 합성합니다.
Track A의 공개 데이터와 혼합하면 일반 한국어 능력 + 도메인 전문성을 동시에 확보할 수 있습니다.

In [ ]:
def chunk_markdown_document(
    text: str, source_doc: str, target_chars: int = 1200
) -> list:
    """마크다운 문서를 heading/크기 기준으로 청킹한다."""
    lines = [line.rstrip() for line in text.splitlines()]
    chunks, current_lines, current_len, chunk_index = [], [], 0, 0

    def flush():
        nonlocal current_lines, current_len, chunk_index
        joined = "\n".join(l for l in current_lines if l.strip()).strip()
        if not joined:
            return
        chunks.append(
            {
                "source_doc": source_doc,
                "chunk_id": f"{source_doc}::chunk_{chunk_index:03d}",
                "chunk_text": joined,
                "char_count": len(joined),
            }
        )
        chunk_index += 1
        current_lines = []
        current_len = 0

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith("#") and current_lines:
            flush()
        if current_len + len(stripped) > target_chars and current_lines:
            flush()
        current_lines.append(stripped)
        current_len += len(stripped)
    flush()
    return chunks


seed_chunks = []
missing_docs = []

for name, path in SEED_DOCS.items():
    if path.exists():
        text = path.read_text(encoding="utf-8")
        doc_chunks = chunk_markdown_document(text, name)
        seed_chunks.extend(doc_chunks)
        print(f"  Loaded '{name}': {len(doc_chunks)} chunks")
    else:
        missing_docs.append(str(path))
        print(f"  [WARN] Not found: {path}")

if missing_docs:
    print(
        f"\n[WARNING] {len(missing_docs)} seed doc(s) missing. "
        "Fewer training examples will be generated."
    )

seed_chunks = seed_chunks[:MAX_SOURCE_CHUNKS]
print(f"\nUsing {len(seed_chunks)} chunks (MAX_SOURCE_CHUNKS={MAX_SOURCE_CHUNKS})")

pd.DataFrame(seed_chunks)[["source_doc", "chunk_id", "char_count"]]

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. Set it via: export OPENAI_API_KEY='sk-...'"
    )

client = OpenAI()
print(f"OpenAI client ready (model={OPENAI_MODEL})")

## Part 1: SFT 데이터 생성

Responses API의 Structured Outputs 기능을 사용해 JSON Schema에 맞는 SFT 행을 생성합니다.

| 필드 | 설명 |
|------|------|
| `instruction` | 한국어 지시문 |
| `input` | 추가 컨텍스트 (없으면 빈 문자열) |
| `output` | 한국어 모범 답변 |
| `source_doc` | 원본 문서 이름 |
| `task_type` | 태스크 유형 (6종) |
| `quality_flags` | 품질 태그 목록 |

In [ ]:
SFT_ROW_SCHEMA = {
    "type": "object",
    "properties": {
        "rows": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "instruction": {"type": "string"},
                    "input": {"type": "string"},
                    "output": {"type": "string"},
                    "source_doc": {"type": "string"},
                    "task_type": {"type": "string"},
                    "quality_flags": {"type": "array", "items": {"type": "string"}},
                },
                "required": [
                    "instruction",
                    "input",
                    "output",
                    "source_doc",
                    "task_type",
                    "quality_flags",
                ],
                "additionalProperties": False,
            },
        }
    },
    "required": ["rows"],
    "additionalProperties": False,
}


def generate_sft_rows(chunk: dict, n_examples: int = SFT_EXAMPLES_PER_CHUNK) -> list:
    """단일 청크에서 SFT 행을 생성한다."""
    system_prompt = (
        "You create Korean SFT dataset rows for AgentOps and observability training. "
        "Every row must stay grounded in the provided source chunk. "
        "Do not invent facts not supported by the source. "
        "Return only schema-compliant JSON."
    )
    user_prompt = (
        f"Source: {chunk['source_doc']} | Chunk: {chunk['chunk_id']}\n"
        f"Allowed task types: {', '.join(DOMAIN_TASK_TYPES)}\n"
        f"Need: {n_examples} Korean SFT rows.\n"
        "Rules: Korean output, practical AgentOps scenarios, "
        "quality_flags must include 'source_grounded' and 'needs_human_review'.\n"
        f"Source chunk:\n{chunk['chunk_text']}"
    )

    resp = client.responses.create(
        model=OPENAI_MODEL,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "sft_gen",
                "schema": SFT_ROW_SCHEMA,
                "strict": True,
            }
        },
    )
    return json.loads(resp.output_text)["rows"]


print("SFT schema and generator defined")

In [ ]:
sft_rows = []

for i, chunk in enumerate(seed_chunks):
    try:
        rows = generate_sft_rows(chunk)
        sft_rows.extend(rows)
        print(f"  [{i + 1}/{len(seed_chunks)}] {chunk['chunk_id']} -> {len(rows)} rows")
    except Exception as e:
        print(f"  [{i + 1}/{len(seed_chunks)}] ERROR on {chunk['chunk_id']}: {e}")
    time.sleep(0.3)

print(f"\nGenerated {len(sft_rows)} SFT rows total")

## Part 2: Embedding 데이터 생성

Instruction-aware 임베딩 학습을 위한 트리플렛(query / positive / hard_negative)을 생성합니다.

| 필드 | 설명 |
|------|------|
| `query` | 한국어 검색 질의 |
| `positive` | 질의에 정확히 답하는 한국어 패시지 |
| `hard_negative` | 유사 주제이지만 오답인 한국어 패시지 |
| `task_instruction` | 영어 태스크 지시문 (instruction-aware 모델용) |
| `source_doc` | 원본 문서 이름 |
| `quality_flags` | 품질 태그 목록 |

> `hard_negative`는 단순히 무관한 문장이 아니라, 동일 도메인의 혼동 가능한 내용이어야 합니다.

In [ ]:
EMB_ROW_SCHEMA = {
    "type": "object",
    "properties": {
        "rows": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "positive": {"type": "string"},
                    "hard_negative": {"type": "string"},
                    "task_instruction": {"type": "string"},
                    "source_doc": {"type": "string"},
                    "quality_flags": {"type": "array", "items": {"type": "string"}},
                },
                "required": [
                    "query",
                    "positive",
                    "hard_negative",
                    "task_instruction",
                    "source_doc",
                    "quality_flags",
                ],
                "additionalProperties": False,
            },
        }
    },
    "required": ["rows"],
    "additionalProperties": False,
}


def generate_embedding_rows(
    chunk: dict, n_examples: int = EMB_EXAMPLES_PER_CHUNK
) -> list:
    """단일 청크에서 임베딩 트리플렛을 생성한다."""
    system_prompt = (
        "You create Korean embedding training data for AgentOps retrieval. "
        "Each row needs: a Korean query, a Korean positive passage (grounded in source), "
        "a Korean hard_negative passage (similar topic but different/wrong answer), "
        "and an English task_instruction for instruction-aware retrieval. "
        "hard_negative must be confusable with the positive, not completely unrelated. "
        "Return only schema-compliant JSON."
    )
    instructions_list = ", ".join(f'"{i}"' for i in EMB_TASK_INSTRUCTIONS)
    user_prompt = (
        f"Source: {chunk['source_doc']} | Chunk: {chunk['chunk_id']}\n"
        f"Choose task_instruction from: [{instructions_list}]\n"
        f"Need: {n_examples} Korean embedding triplets.\n"
        "Rules: query/positive/hard_negative in Korean, task_instruction in English. "
        "quality_flags must include 'source_grounded' and 'needs_human_review'.\n"
        f"Source chunk:\n{chunk['chunk_text']}"
    )

    resp = client.responses.create(
        model=OPENAI_MODEL,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "emb_gen",
                "schema": EMB_ROW_SCHEMA,
                "strict": True,
            }
        },
    )
    return json.loads(resp.output_text)["rows"]


print("Embedding schema and generator defined")

In [ ]:
emb_rows = []

for i, chunk in enumerate(seed_chunks):
    try:
        rows = generate_embedding_rows(chunk)
        emb_rows.extend(rows)
        print(f"  [{i + 1}/{len(seed_chunks)}] {chunk['chunk_id']} -> {len(rows)} rows")
    except Exception as e:
        print(f"  [{i + 1}/{len(seed_chunks)}] ERROR on {chunk['chunk_id']}: {e}")
    time.sleep(0.3)

print(f"\nGenerated {len(emb_rows)} embedding rows total")

## Part 3: 품질 검수

생성된 데이터에 대해 다음 검사를 수행합니다.

1. **필수 필드 유무** — instruction/output/query/positive 등이 비어 있으면 제외
2. **source_doc 유효성** — seed 문서 키와 일치하는지 확인
3. **한국어 비율** — 출력 텍스트에서 한국어 문자 비율 < 20%이면 제외
4. **도메인 키워드** — SFT 행에 도메인 키워드가 전혀 없으면 검토 플래그
5. **중복 제거** — (instruction, output) 또는 (query, positive) 쌍 기준 중복 제거

In [ ]:
def korean_char_ratio(text: str) -> float:
    """텍스트에서 한국어 문자(가-힣) 비율을 반환한다."""
    text = text or ""
    total = sum(1 for c in text if not c.isspace())
    if total == 0:
        return 0.0
    return sum(1 for c in text if "가" <= c <= "힣") / total


def domain_keyword_hits(text: str) -> list:
    """텍스트에서 발견된 도메인 키워드 목록을 반환한다."""
    text = text.lower()
    return sorted(
        set(kw for kwlist in DOMAIN_KEYWORDS.values() for kw in kwlist if kw in text)
    )


def dedupe_rows(rows: list, key_fn) -> list:
    """key_fn 기준으로 중복 행을 제거한다."""
    seen, result = set(), []
    for r in rows:
        k = key_fn(r)
        if k not in seen:
            seen.add(k)
            result.append(r)
    return result


print("Quality-check helpers defined")

In [ ]:
VALID_SOURCES = set(SEED_DOCS.keys())
VALID_TASKS = set(DOMAIN_TASK_TYPES)


def check_sft_row(row: dict) -> list:
    """SFT 행의 품질 문제 목록을 반환한다. 빈 리스트 = 합격."""
    issues = []
    if not row.get("instruction", "").strip():
        issues.append("missing_instruction")
    if not row.get("output", "").strip():
        issues.append("missing_output")
    if row.get("source_doc") not in VALID_SOURCES:
        issues.append("invalid_source_doc")
    if row.get("task_type") not in VALID_TASKS:
        issues.append("invalid_task_type")
    combined = f"{row.get('instruction', '')} {row.get('output', '')}"
    if korean_char_ratio(combined) < 0.2:
        issues.append("low_korean")
    if not domain_keyword_hits(combined):
        issues.append("no_domain_keyword")
    return issues


# 중복 제거 후 품질 검사
sft_rows = dedupe_rows(
    sft_rows, lambda r: (r.get("instruction", "").strip(), r.get("output", "").strip())
)
sft_checked = [{"row": r, "issues": check_sft_row(r)} for r in sft_rows]
sft_accept = [x["row"] for x in sft_checked if not x["issues"]]
sft_review = [x for x in sft_checked if x["issues"]]

print(f"SFT: {len(sft_accept)} accepted, {len(sft_review)} need review")
if sft_review:
    print("  Sample issues:")
    for item in sft_review[:3]:
        print(f"    {item['issues']} <- {item['row'].get('instruction', '')[:60]}")

In [ ]:
def check_emb_row(row: dict) -> list:
    """임베딩 행의 품질 문제 목록을 반환한다. 빈 리스트 = 합격."""
    issues = []
    if not row.get("query", "").strip():
        issues.append("missing_query")
    if not row.get("positive", "").strip():
        issues.append("missing_positive")
    if not row.get("hard_negative", "").strip():
        issues.append("missing_hard_negative")
    if not row.get("task_instruction", "").strip():
        issues.append("missing_instruction")
    if row.get("source_doc") not in VALID_SOURCES:
        issues.append("invalid_source_doc")
    combined = f"{row.get('query', '')} {row.get('positive', '')}"
    if korean_char_ratio(combined) < 0.2:
        issues.append("low_korean")
    if korean_char_ratio(row.get("task_instruction", "")) > 0.1:
        issues.append("korean_instruction")
    return issues


# 중복 제거 후 품질 검사
emb_rows = dedupe_rows(
    emb_rows, lambda r: (r.get("query", "").strip(), r.get("positive", "").strip())
)
emb_checked = [{"row": r, "issues": check_emb_row(r)} for r in emb_rows]
emb_accept = [x["row"] for x in emb_checked if not x["issues"]]
emb_review = [x for x in emb_checked if x["issues"]]

print(f"Embedding: {len(emb_accept)} accepted, {len(emb_review)} need review")
if emb_review:
    print("  Sample issues:")
    for item in emb_review[:3]:
        print(f"    {item['issues']} <- {item['row'].get('query', '')[:60]}")

In [ ]:
print("=== SFT 미리보기 ===")
if sft_accept:
    display(
        pd.DataFrame(sft_accept[:5])[
            ["instruction", "output", "source_doc", "task_type"]
        ]
    )
else:
    print("  (합격 행 없음 — 생성 파라미터를 확인하세요)")

print("\n=== Embedding 미리보기 ===")
if emb_accept:
    display(
        pd.DataFrame(emb_accept[:5])[
            ["query", "positive", "hard_negative", "task_instruction"]
        ]
    )
else:
    print("  (합격 행 없음 — 생성 파라미터를 확인하세요)")

---

## Part 5: 데이터셋 혼합 — Track A + Track B 통합

Track A(공개 데이터)와 Track B(합성 데이터)를 혼합합니다.

### 혼합 비율 전략

| 구성 요소 | 역할 | 비율 |
|-----------|------|------|
| **일반 한국어** (공개 데이터 랜덤 샘플) | 한국어 응답 능력 유지 | 60% |
| **도메인 인접** (공개 데이터 키워드 필터) | IT/운영 관련 기초 지식 | 20% |
| **도메인 전용** (Track B 합성 데이터) | AgentOps 전문성 | 20% |

> Track B를 건너뛴 경우: 일반 70% + 도메인 인접 30%로 자동 조정됩니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Part 5-1: SFT 데이터 혼합
# ══════════════════════════════════════════════════════════════════════════════

# Track B 합성 데이터가 있는지 확인
has_synthetic_sft = "sft_accept" in dir() and len(sft_accept) > 0

if has_synthetic_sft:
    print(f"Track B 합성 SFT 데이터: {len(sft_accept)}건")
    # 3-Way 혼합: 일반 60% + 도메인 인접 20% + 합성 20%
    n_synthetic = len(sft_accept)
    n_adjacent = max(100, n_synthetic)  # 합성과 비슷한 양
    n_general = max(200, n_synthetic * 3)  # 일반은 3배
else:
    print("Track B를 건너뛰었습니다. Track A 데이터만으로 혼합합니다.")
    # 2-Way 혼합: 일반 70% + 도메인 인접 30%
    n_adjacent = min(
        1500, len(sft_base_domain) + len(sft_human_domain) + len(sft_dolly_domain)
    )
    n_general = min(3500, n_adjacent * 2)

# ── 일반 한국어 샘플 (랜덤) ──
sft_general_pool = sft_base.shuffle(seed=SEED).select(
    range(min(n_general, len(sft_base)))
)


# ── 도메인 인접 샘플 (키워드 필터링) ──
# 세 데이터셋의 도메인 인접 샘플을 합치고, instruction/output만 추출
def normalize_sft_columns(ds, source_name):
    """다양한 스키마를 instruction/input/output으로 통일한다."""
    rows = []
    for ex in ds:
        rows.append(
            {
                "instruction": ex.get("instruction", ex.get("question", "")),
                "input": ex.get("input", ex.get("context", "")),
                "output": ex.get("output", ex.get("response", "")),
                "source_doc": source_name,
                "task_type": "public_domain_adjacent",
                "quality_flags": ["public_dataset", "domain_filtered"],
            }
        )
    return rows


adjacent_rows = []
adjacent_rows.extend(normalize_sft_columns(sft_base_domain, "korean_rlhf"))
adjacent_rows.extend(normalize_sft_columns(sft_human_domain, "koalpaca"))
adjacent_rows.extend(normalize_sft_columns(sft_dolly_domain, "dolly_15k_ko"))
random.shuffle(adjacent_rows)
adjacent_rows = adjacent_rows[:n_adjacent]

# ── 일반 샘플도 동일 스키마로 변환 ──
general_rows = normalize_sft_columns(sft_general_pool, "korean_rlhf_general")

# ── 최종 혼합 ──
sft_mixed = general_rows + adjacent_rows
if has_synthetic_sft:
    sft_mixed.extend(sft_accept)
random.shuffle(sft_mixed)

print(f"\n=== SFT 최종 혼합 결과 ===")
print(f"  일반 한국어:   {len(general_rows):,}건")
print(f"  도메인 인접:   {len(adjacent_rows):,}건")
if has_synthetic_sft:
    print(f"  도메인 합성:   {len(sft_accept):,}건")
print(f"  합계:          {len(sft_mixed):,}건")

# 소스 분포 확인
from collections import Counter

source_dist = Counter(r.get("source_doc", "unknown") for r in sft_mixed)
print(f"\n소스 분포: {dict(source_dist)}")

# 미리보기
display(pd.DataFrame(sft_mixed[:5])[["instruction", "output", "source_doc"]])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Part 5-2: Embedding 데이터 혼합
# ══════════════════════════════════════════════════════════════════════════════

has_synthetic_emb = "emb_accept" in dir() and len(emb_accept) > 0

if has_synthetic_emb:
    print(f"Track B 합성 Embedding 데이터: {len(emb_accept)}건")
    n_public_emb = max(2000, len(emb_accept) * 5)  # 합성의 5배
else:
    print("Track B를 건너뛰었습니다. Track A 데이터만으로 혼합합니다.")
    n_public_emb = 5000

# ── ko-triplet에서 서브샘플 + 컬럼명 통일 ──
emb_triplet_sample = emb_triplet.shuffle(seed=SEED).select(
    range(min(n_public_emb, len(emb_triplet)))
)

emb_public_rows = []
for ex in emb_triplet_sample:
    emb_public_rows.append(
        {
            "query": ex["query"],
            "positive": ex["document"],
            "hard_negative": ex["hard_negative"],
            "task_instruction": "Retrieve relevant Korean passages",
            "source_doc": "ko-triplet-v1.0",
            "quality_flags": ["public_dataset"],
        }
    )

# ── NLI SimCSE에서 서브샘플 (선택적 확장) ──
NLI_SAMPLE_SIZE = min(1000, len(emb_nli))
emb_nli_sample = emb_nli.shuffle(seed=SEED).select(range(NLI_SAMPLE_SIZE))

emb_nli_rows = []
for ex in emb_nli_sample:
    emb_nli_rows.append(
        {
            "query": ex["premise"],
            "positive": ex["entailment"],
            "hard_negative": ex["contradiction"],
            "task_instruction": "Find the sentence with the same meaning",
            "source_doc": "kor_nli_simcse",
            "quality_flags": ["public_dataset", "nli_converted"],
        }
    )

# ── KorQuAD + Hard Negative에서 서브샘플 ──
KORQUAD_SAMPLE_SIZE = min(1000, len(emb_korquad))
emb_korquad_sample = emb_korquad.shuffle(seed=SEED).select(range(KORQUAD_SAMPLE_SIZE))

emb_korquad_rows = []
for ex in emb_korquad_sample:
    negs = ex.get("negative_samples", [])
    if negs and ex.get("context"):
        emb_korquad_rows.append(
            {
                "query": ex["question"],
                "positive": ex["context"],
                "hard_negative": negs[0],  # 가장 어려운 negative
                "task_instruction": "Retrieve the passage that answers this question",
                "source_doc": "korquad_neg",
                "quality_flags": ["public_dataset", "bm25_negative"],
            }
        )

# ── 최종 혼합 ──
emb_mixed = emb_public_rows + emb_nli_rows + emb_korquad_rows
if has_synthetic_emb:
    emb_mixed.extend(emb_accept)
random.shuffle(emb_mixed)

print(f"\n=== Embedding 최종 혼합 결과 ===")
print(f"  ko-triplet:    {len(emb_public_rows):,}건 (범용 검색 트리플릿)")
print(f"  NLI SimCSE:    {len(emb_nli_rows):,}건 (의미 유사도)")
print(f"  KorQuAD+neg:   {len(emb_korquad_rows):,}건 (QA 검색)")
if has_synthetic_emb:
    print(f"  도메인 합성:   {len(emb_accept):,}건 (AgentOps 전용)")
print(f"  합계:          {len(emb_mixed):,}건")

# 소스 분포
source_dist = Counter(r.get("source_doc", "unknown") for r in emb_mixed)
print(f"\n소스 분포: {dict(source_dist)}")

display(
    pd.DataFrame(emb_mixed[:5])[["query", "positive", "hard_negative", "source_doc"]]
)

In [ ]:
def write_jsonl(path: Path, rows: list) -> None:
    """행 목록을 JSONL 파일로 저장한다."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  Saved {len(rows)} rows -> {path}")


# ══════════════════════════════════════════════════════════════════════════════
# Part 6: 혼합 데이터 → Train/Eval 분리 → 저장
# ══════════════════════════════════════════════════════════════════════════════

# ── SFT 분리 ─────────────────────────────────────────────────────────────────
random.shuffle(sft_mixed)
sft_eval_size = min(40, max(10, int(len(sft_mixed) * 0.05)))  # 5% (최대 40건)
sft_eval_final = sft_mixed[:sft_eval_size]
sft_train_final = sft_mixed[sft_eval_size:]

write_jsonl(SFT_TRAIN_PATH, sft_train_final)
write_jsonl(SFT_EVAL_PATH, sft_eval_final)

# ── Embedding 분리 ────────────────────────────────────────────────────────────
random.shuffle(emb_mixed)
emb_eval_size = min(NUM_EVAL_QUERIES, max(10, int(len(emb_mixed) * 0.05)))
emb_eval_final = emb_mixed[:emb_eval_size]
emb_train_final = emb_mixed[emb_eval_size:]

for r in emb_train_final:
    r["split"] = "train"
for r in emb_eval_final:
    r["split"] = "eval"

EMB_DATA_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(EMB_TRAIN_PATH, emb_train_final)
write_jsonl(EMB_EVAL_PATH, emb_eval_final)

# ── 최종 요약 ─────────────────────────────────────────────────────────────────
summary = pd.DataFrame(
    [
        {
            "file": str(SFT_TRAIN_PATH),
            "rows": len(sft_train_final),
            "purpose": "Lab 1 학습",
        },
        {
            "file": str(SFT_EVAL_PATH),
            "rows": len(sft_eval_final),
            "purpose": "Lab 1 평가",
        },
        {
            "file": str(EMB_TRAIN_PATH),
            "rows": len(emb_train_final),
            "purpose": "Lab 2 학습",
        },
        {
            "file": str(EMB_EVAL_PATH),
            "rows": len(emb_eval_final),
            "purpose": "Lab 2 평가",
        },
    ]
)
print("\n=== 최종 저장 결과 ===")
display(summary)
print(f"\n총 {summary['rows'].sum():,}건 저장 완료. Lab 1/Lab 2로 이동하세요.")

## 다음 단계

데이터 구축이 완료되었습니다! 아래 Lab으로 이동하세요.

| Lab | 사용 파일 | 내용 |
|-----|-----------|------|
| **Lab 1** | `sft_train.jsonl`, `sft_eval.jsonl` | Qwen3.5-4B SFT |
| **Lab 2** | `data/embedding_train.jsonl`, `data/embedding_eval.jsonl` | Qwen3-Embedding-0.6B 튜닝 |

### 데이터를 더 늘리고 싶다면

```python
# Track A: 공개 데이터 서브샘플 크기 조정
n_general = 10000          # 일반 한국어 (기본: 3000)
n_public_emb = 20000       # ko-triplet 서브셋 (기본: 5000)
NLI_SAMPLE_SIZE = 5000     # NLI SimCSE (기본: 1000)
KORQUAD_SAMPLE_SIZE = 3000 # KorQuAD+neg (기본: 1000)

# Track B: 합성 데이터 파라미터 조정
MAX_SOURCE_CHUNKS = 12     # 더 많은 청크
SFT_EXAMPLES_PER_CHUNK = 5 # 청크당 SFT 예시
EMB_EXAMPLES_PER_CHUNK = 6 # 청크당 임베딩 예시
```

### 사용된 공개 데이터셋 출처

| 데이터셋 | 크기 | 라이선스 | 용도 |
|----------|------|----------|------|
| `jojo0217/korean_rlhf_dataset` | 107K | Apache 2.0 | SFT 일반+도메인 인접 |
| `beomi/KoAlpaca-v1.1a` | 21K | - | SFT 도메인 인접 (인간 QA) |
| `nlpai-lab/databricks-dolly-15k-ko` | 15K | CC-BY-SA 3.0 | SFT 다양한 태스크 |
| `nlpai-lab/ko-triplet-v1.0` | 744K | - | Embedding 트리플릿 |
| `dkoterwa/kor_nli_simcse` | 487K | CC BY-SA 4.0 | Embedding NLI→SimCSE |
| `sungmineom/korquad_negative_samples` | 147K | - | Embedding QA+neg |